In [1]:
# importing necessary libraries

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Polygon, Point
from shapely.affinity import scale
from scipy.special import gamma
import warnings
warnings.filterwarnings('ignore')

from py_wake.site import XRSite
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine
from py_wake.wind_turbines import WindTurbines
from py_wake.literature import Nygaard_2022

In [2]:
# file paths

DATA_DIR = "../data"
ERA5_DIR = "../data/era5"
OUTPUT_DIR = "../data/train"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_SAMPLE_SCENARIOS = 200
RANDOM_SEED = 42
 
# Target farm features
TARGET_N_TURBINES = 44
TARGET_ROTOR_DIA_M = 164
TARGET_HUB_HEIGHT_M = 105
TARGET_RATED_POWER = 8400  # kW
TARGET_SPACING_D = 5

# Neighbor farm scenarios
NB_DISTANCES_KM = [5, 50]
NB_DIRECTION_DEG = np.arange(0, 360, 30)  # 12 directions
NB_SIZES = [20, 200]
NB_SPACINGS_D = [4, 12]
TI_VALUES = [0.04, 0.1]

# Wind conditions to simulate
WS_SIM = (4.0, 15.0)
WD_SIM = np.arange(0, 360, 30)

# Neighbour turbine specs
NEIGHBOUR_SPECS = [
    {
        'name': 'V80-2.0',
        'diameter': 80,
        'hub_height': 70,
        'rated_power': 2000
    },
    {
        'name': 'SWT-3.6-107',
        'diameter': 107,
        'hub_height': 80,
        'rated_power': 3600
    },
    {
        'name': 'V112-3.45',
        'diameter': 112,
        'hub_height': 84,
        'rated_power': 3450
    },
    {
        'name': 'SWT-6.0-154',
        'diameter': 154,
        'hub_height': 100,
        'rated_power': 6000
    },
    {
        'name': 'SG-7.0-154',
        'diameter': 154,
        'hub_height': 105,
        'rated_power': 7000
    },
    {
        'name': 'SG-8.0-167',
        'diameter': 167,
        'hub_height': 105,
        'rated_power': 8400
    },
]

# Jensen wake expansion parameter (offshore)
KW_JENSEN = 0.05

FARM_CONFIGS = [
    # ---- LARGE FARMS (100+ turbines) ----
    {
        'name': 'London Array',
        'n_turbines': 175,
        'rotor_dia': 107,
        'hub_height': 87,
        'rated_power': 3600,
        'spacing_D': 6,
        'epsg': 32631,
    },
    {
        'name': 'Hornsea 1',
        'n_turbines': 174,
        'rotor_dia': 154,
        'hub_height': 100,
        'rated_power': 7000,
        'spacing_D': 8,
        'epsg': 32631,
    },
    {
        'name': 'Anholt',
        'n_turbines': 111,
        'rotor_dia': 120,
        'hub_height': 82,
        'rated_power': 3600,
        'spacing_D': 6,
        'epsg': 32632,
    },
    {
        'name': 'Rampion',
        'n_turbines': 116,
        'rotor_dia': 112,
        'hub_height': 84,
        'rated_power': 3450,
        'spacing_D': 5,
        'epsg': 32630,
    },
    # ---- MEDIUM FARMS (50-100 turbines) ----
    {
        'name': 'Walney Extension',
        'n_turbines': 87,
        'rotor_dia': 154,
        'hub_height': 105,
        'rated_power': 7000,
        'spacing_D': 9,
        'epsg': 32630,
    },
    {
        'name': 'DanTysk',
        'n_turbines': 80,
        'rotor_dia': 120,
        'hub_height': 88,
        'rated_power': 3600,
        'spacing_D': 6,
        'epsg': 32632,
    },
    {
        'name': 'Saint-Nazaire',
        'n_turbines': 80,
        'rotor_dia': 154,
        'hub_height': 100,
        'rated_power': 7000,
        'spacing_D': 8,
        'epsg': 32630,
    },
    {
        'name': 'Race Bank',
        'n_turbines': 91,
        'rotor_dia': 154,
        'hub_height': 95,
        'rated_power': 6000,
        'spacing_D': 7,
        'epsg': 32631,
    },
    {
        'name': 'Kriegers Flak',
        'n_turbines': 72,
        'rotor_dia': 167,
        'hub_height': 100,
        'rated_power': 8400,
        'spacing_D': 9,
        'epsg': 32633,
    },
    {
        'name': 'Dudgeon',
        'n_turbines': 67,
        'rotor_dia': 154,
        'hub_height': 95,
        'rated_power': 6000,
        'spacing_D': 8,
        'epsg': 32631,
    },
    # ---- SMALL FARMS (<50 turbines) ----
    {
        'name': 'Norther',
        'n_turbines': 44,
        'rotor_dia': 164,
        'hub_height': 105,
        'rated_power': 8400,
        'spacing_D': 10,
        'epsg': 32631,
    },
]

TEST_FARM_CONFIG = [{'name': 'Horns Rev I',
                     'n_turbines': 80,
                     'rotor_dia': 80,
                     'hub_height': 70,
                     'rated_power': 2000,
                     'spacing_D': 7,
                     'epsg': 32631,}]
                     

In [3]:
# Load ERA5 wind rose

def load_era5_wind_rose(data_dir):
    from scipy.stats import weibull_min

    ds = xr.open_mfdataset(os.path.join(data_dir, 'era5_*.nc'))
    u = ds['u100'].values.flatten()
    v = ds['v100'].values.flatten()

    ws = np.sqrt(u**2 + v**2)
    wd = (270 - np.degrees(np.arctan2(v, u))) % 360

    sectors = np.arange(0, 360, 30)
    rows = []

    for sector in sectors:
        lower = (sector - 15) % 360
        upper = (sector + 15) % 360

        if lower > upper:
            mask = (wd >= lower) | (wd < upper)
        else:
            mask = (wd >= lower) & (wd < upper)

        ws_sector = ws[mask]
        ws_sector = ws_sector[ws_sector > 0.5]

        freq  = mask.sum() / len(ws)
        K, _, A = weibull_min.fit(ws_sector, floc=0)
        mean_ws_sector = A * gamma(1+1/K)

        rows.append({'direction_deg': sector,
            'frequency': freq,
            'weibull_A': A,
            'weibull_k': K,
            'mean_ws': mean_ws_sector,
        })

    wind_rose_df = pd.DataFrame(rows)
    wind_rose_df['frequency'] = wind_rose_df['frequency'] / wind_rose_df['frequency'].sum()
    return wind_rose_df

    

In [4]:
wind_rose = load_era5_wind_rose(ERA5_DIR)
print(wind_rose)

    direction_deg  frequency  weibull_A  weibull_k    mean_ws
0               0   0.056037   8.472460   2.294175   7.505663
1              30   0.081851   8.867088   2.797608   7.895451
2              60   0.071244   8.886848   2.641675   7.897245
3              90   0.056037   8.156268   2.536340   7.239430
4             120   0.053224   7.255386   2.426867   6.433157
5             150   0.048054   7.803432   2.173249   6.910752
6             180   0.090062  10.556416   2.323428   9.353252
7             210   0.128574  11.538733   2.416911  10.230249
8             240   0.197004  11.900682   2.712951  10.584818
9             270   0.099034  10.339494   2.162859   9.156691
10            300   0.065845   9.510562   2.055468   8.425087
11            330   0.053034   8.554198   2.037782   7.578702


#### Load Target Farm

In [5]:
# Load Norther farm polygon
gdf = gpd.read_file(os.path.join(DATA_DIR, 'candidate_farms.gpkg'))

if gdf.crs.is_geographic:
    gdf = gdf.to_crs(epsg=32361)

target = gdf[gdf['name'].str.contains('Norther')].iloc[0]
target_poly = target.geometry
target_centroid = target.geometry.centroid

print(target_poly)
print(target_centroid)

MULTIPOLYGON (((502777.299591349 5706764.814813434, 498146.67055414955 5704261.765602958, 495614.7662000849 5706861.375850357, 496189.3362810096 5707130.939097753, 498126.3158488074 5708550.055429196, 498531.6487974907 5709208.778312305, 499096.8194067634 5709224.767747555, 499388.42951208877 5709483.091081421, 499935.052862206 5709995.546213035, 500389.51122725743 5710511.022534289, 500785.4375232379 5710934.892788898, 501178.4172328713 5711234.912766728, 501744.57991305966 5711531.201790891, 502141.9453942678 5711847.844168637, 502309.10005110543 5712385.441768087, 502046.2485648827 5713782.866801226, 505204.4121625023 5710726.724481557, 505723.0040016876 5710301.332223273, 502777.299591349 5706764.814813434)))
POINT (500911.69264268246 5708425.632954922)


In [6]:
# function to palce turbines inside the polygon
def place_turbines(polygon, n_turbines, min_spacing_m, seed, max_expand_attempts=3):
    current_poly = polygon
    rng = np.random.default_rng(seed)
    
    for expand_attempt in range(max_expand_attempts+1):
        min_x, min_y, max_x, max_y = current_poly.bounds
        x_placed, y_placed = [], []
        attempts = 0
        max_attempts = n_turbines * 200

        while len(x_placed) < n_turbines and attempts < max_attempts:
            x = rng.uniform(min_x, max_x)
            y = rng.uniform(min_y, max_y)

            if not current_poly.contains(Point(x, y)):
                attempts +=1
                continue

            too_close = False
            for xi, yi in zip(x_placed, y_placed):
                if (x - xi)**2 + (y - yi)**2 < min_spacing_m**2:
                    too_close = True
                    break
            if not too_close:
                x_placed.append(x)
                y_placed.append(y)

            attempts +=1

        if len(x_placed) == n_turbines:
            return np.array(x_placed), np.array(y_placed), current_poly

        if expand_attempt < max_expand_attempts:
            centroid  = current_poly.centroid
            current_poly = scale(current_poly, xfact=1.25, yfact=1.25)

    return np.array(x_placed), np.array(y_placed), current_poly

In [7]:
# Function to create neighbour polygons

def make_neighbour_polygon(centroid_x, centroid_y, distance_km, direction_deg, 
                           n_turbines, spacing_D, rotor_dia):
    turbines_per_side = int(np.ceil(np.sqrt(n_turbines)))
    spacing_m = spacing_D * rotor_dia
    side_length = (turbines_per_side + 1) * spacing_m

    # Neighbour centroid position
    distance_m = distance_km * 1000
    direction_rad = np.radians(direction_deg)

    nb_x = centroid_x + distance_m * np.sin(direction_rad)
    nb_y = centroid_y + distance_m * np.cos(direction_rad)

    poly = Polygon([(nb_x - side_length/2, nb_y - side_length/2),
                    (nb_x + side_length/2, nb_y - side_length/2),
                    (nb_x + side_length/2, nb_y + side_length/2),
                    (nb_x - side_length/2, nb_y + side_length/2)])

    return poly, nb_x, nb_y
    

In [8]:
# Extract geometric features
def compute_geometric_features(target_x, target_y, target_rotor_dia, 
                               neighbour_x,neighbour_y, neighbour_rotor_dia,
                               wind_direction_deg, kw=KW_JENSEN):
    
    wind_going_rad = np.radians(wind_direction_deg + 180)

    ux = np.sin(wind_going_rad)
    uy = np.cos(wind_going_rad)

    dx = target_x - neighbour_x
    dy = target_y - neighbour_y

    streamwise_dst = dx * ux + dy *uy
    lateral_dst = np.abs(dx * (-uy) + (dy * ux))

    target_radius = target_rotor_dia / 2
    upstream_mask = streamwise_dst > 0

    blocking_radius_geom = neighbour_rotor_dia / 2 + target_rotor_dia
    blocking_mask_geom = upstream_mask & (lateral_dst < blocking_radius_geom)
    
    geom_features = compute_blocking_features(blocking_mask_geom, streamwise_dst, lateral_dst, neighbour_rotor_dia, target_rotor_dia, prefix='geom')

    # Physics-informed features

    Dw = np.where(upstream_mask, neighbour_rotor_dia + 2 * kw * streamwise_dst, neighbour_rotor_dia)
    blocking_radius_pi = Dw / 2 + target_radius
    blocking_mask_pi = upstream_mask & (lateral_dst < blocking_radius_pi)

    pi_feats = compute_blocking_features(blocking_mask_pi, streamwise_dst, lateral_dst, neighbour_rotor_dia, target_rotor_dia, prefix='pi')
    
    features = {}
    features.update(geom_features)
    features.update(pi_feats)
    return features

In [9]:
def compute_blocking_features(blocking_mask, streamwise_dst, lateral_dst, neighbour_rotor_dia,
                              target_rotor_dia, prefix):
    n_blocking = int(blocking_mask.sum())

    if n_blocking == 0:
        return {
            f'{prefix}_n_blocking': 0,
            f'{prefix}_blocking_ratio': 0.0,
            f'{prefix}_blocking_distance': 0.0,
            f'{prefix}_streamwise_dist_1st': 0.0,
            f'{prefix}_lateral_dist_1st': 0.0,
            f'{prefix}_rotor_dia_1st': 0.0,
        }
        
    blocking_streamwise = streamwise_dst[blocking_mask]
    blocking_lateral = lateral_dst[blocking_mask]
    blocking_rotor_dia = neighbour_rotor_dia[blocking_mask]

    # blocking ratio
    overlap_per_turbine = np.clip((blocking_rotor_dia/2 + target_rotor_dia/2 - blocking_lateral) / target_rotor_dia, 0, 1)
    blocking_ratio = float(np.clip(overlap_per_turbine.sum(), 0, 1))

    # Blocking distance: mean streamwise distance of blockers
    blocking_distance = float(blocking_streamwise.mean())
    idx_closest = np.argmin(blocking_streamwise)

    return {
        f'{prefix}_n_blocking': n_blocking,
        f'{prefix}_blocking_ratio': blocking_ratio,
        f'{prefix}_blocking_distance': blocking_distance,
        f'{prefix}_streamwise_dist_1st': float(blocking_streamwise[idx_closest]),
        f'{prefix}_lateral_dist_1st': float(blocking_lateral[idx_closest]),
        f'{prefix}_rotor_dia_1st': float(blocking_rotor_dia[idx_closest]),
    }

In [10]:
def sample_scenarios(nb_distance_km, nb_direction_deg, nb_size, nb_spacing_D,
                     ti_values, neighbour_specs, n_samples, ws_range=(3.0, 25.0),
                     wd_range=(0, 360), random_seed=RANDOM_SEED):
    rng = np.random.default_rng(random_seed)

    scenarios = []

    for i in range(n_samples):
        nb_spec = neighbour_specs[rng.integers(len(neighbour_specs))]
        scenario = {
            'distance_km': rng.uniform(nb_distance_km[0], nb_distance_km[-1]),
            'direction_deg': rng.choice(nb_direction_deg),
            'n_nb_turbines': int(rng.uniform(nb_size[0], nb_size[-1])),
            'nb_spacing_D': int(rng.uniform(nb_spacing_D[0], nb_spacing_D[-1])),
            'ti': rng.uniform(0.04, 0.1),
            'nb_spec': nb_spec,
            'nb_rotor_dia_m': nb_spec['diameter'],
            'nb_hub_height_m': nb_spec['hub_height'],
            'nb_rated_power_kw': nb_spec['rated_power'],
            'ws': float(rng.uniform(ws_range[0], ws_range[1])),
            'wd': rng.choice(wd_range)        
        }
        scenarios.append(scenario)

    return scenarios

In [11]:

scenarios = sample_scenarios(
    NB_DISTANCES_KM, NB_DIRECTION_DEG, NB_SIZES, NB_SPACINGS_D,
    TI_VALUES, NEIGHBOUR_SPECS, N_SAMPLE_SCENARIOS, WS_SIM, WD_SIM, RANDOM_SEED)

print(len(scenarios))

200


In [12]:
scenarios_hornsrev = sample_scenarios(
    NB_DISTANCES_KM, NB_DIRECTION_DEG, NB_SIZES, NB_SPACINGS_D,
    TI_VALUES, NEIGHBOUR_SPECS, 15, WS_SIM, WD_SIM, RANDOM_SEED)

print(len(scenarios_hornsrev))

15


In [13]:
def generate_training_data(wind_rose_df, target_poly, target_centroid, target_spec,scenarios, random_seed=42):
    target_spacing_m = TARGET_SPACING_D * target_spec['diameter']
    x_target, y_target, _ = place_turbines(target_poly, target_spec['n_turbines'], target_spacing_m, seed=random_seed)
    n_target = len(x_target)
    print(f"Target farm: {n_target} turbines placed")
    print(f"  x range: [{x_target.min():.0f}, {x_target.max():.0f}] m")
    print(f"  y range: [{y_target.min():.0f}, {y_target.max():.0f}] m")
    
    # Create target wind turbine object
    target_wt = GenericWindTurbine(
        'Target', target_spec['diameter'],
        target_spec['hub_height'], target_spec['rated_power'])
    
    all_results = []
    
    for i, sc in enumerate(scenarios):
        try:
            spec = sc['nb_spec']

            nb_poly, nb_cx, nb_cy = make_neighbour_polygon(
                target_centroid.x, target_centroid.y,
                sc['distance_km'], sc['direction_deg'],
                sc['n_nb_turbines'], sc['nb_spacing_D'], spec['diameter'])

            nb_spacing_m = sc['nb_spacing_D'] * spec['diameter']
            x_nb, y_nb, _ = place_turbines(
                nb_poly, sc['n_nb_turbines'], nb_spacing_m,
                seed=random_seed + i)
            
            if len(x_nb) < sc['n_nb_turbines'] * 0.8:
                continue  # skip if placement failed
            
            nb_wt = GenericWindTurbine(
                'Neighbour', spec['diameter'],
                spec['hub_height'], spec['rated_power'])
            wts = WindTurbines.from_WindTurbine_lst([target_wt, nb_wt])
            
            types_all = [0] * n_target + [1] * len(x_nb)
            x_all = np.concatenate([x_target, x_nb])
            y_all = np.concatenate([y_target, y_nb])
            D_all = np.array(
                [target_spec['diameter']] * n_target +
                [spec['diameter']] * len(x_nb))
            
            # Create site with wind rose
            ds = xr.Dataset(
                data_vars={
                    'Sector_frequency': ('wd', wind_rose_df['frequency'].values),
                    'Weibull_A': ('wd', wind_rose_df['weibull_A'].values),
                    'Weibull_k': ('wd', wind_rose_df['weibull_k'].values),
                    'TI': sc['ti'],
                },
                coords={'wd': wind_rose_df['direction_deg'].values}
            )
            site = XRSite(ds)
            wfm = Nygaard_2022(site, wts)
            
            # Run simulation — get per-turbine power
            sim = wfm(x_all, y_all, type=types_all, wd=[sc['wd']], ws=[sc['ws']])
            
            power_array = sim.Power.values  # Watts
            
            for t_idx in range(n_target):
                # All OTHER turbines (same farm + neighbours, excluding self)
                other_mask = np.ones(len(x_all), dtype=bool)
                other_mask[t_idx] = False
                other_x = x_all[other_mask]
                other_y = y_all[other_mask]
                other_D = D_all[other_mask]
                
                # Geometric features for this wind direction
                geo_feats = compute_geometric_features(
                    x_all[t_idx], y_all[t_idx], D_all[t_idx],
                    other_x, other_y, other_D,
                    sc['wd'], kw=KW_JENSEN)

                # FIX: Index as [t_idx, 0, 0] — only 1 wd and 1 ws
                power_kw = float(power_array[t_idx, 0, 0]) / 1000

                row = {
                    # IDs (for traceability, not ML features)
                    'scenario_id': i,
                    'turbine_idx': t_idx,

                    # Atmospheric inputs
                    'ws_free': float(sc['ws']),
                    'wd': float(sc['wd']),
                    'ti': sc['ti'],

                    # Target turbine specs
                    'target_rotor_dia_m': target_spec['diameter'],
                    'target_hub_height_m': target_spec['hub_height'],
                    'target_rated_power_kw': target_spec['rated_power'],

                    # Neighbour farm specs
                    'nb_distance_km': sc['distance_km'],
                    'nb_direction_deg': sc['direction_deg'],
                    'nb_n_turbines': len(x_nb),
                    'nb_spacing_D': sc['nb_spacing_D'],
                    'nb_rotor_dia_m': spec['diameter'],
                    'nb_hub_height_m': spec['hub_height'],
                    'nb_rated_power_kw': spec['rated_power'],

                    # TARGET VARIABLE
                    'power_kw': power_kw,
                }

                # Add 12 geometric features
                row.update(geo_feats)

                all_results.append(row)
        
        except Exception as e:
            print(f"Scenario {i}: {e}")
            raise e
            continue
    
   
    
    df = pd.DataFrame(all_results)
    return df
                            

In [14]:
# target_spec = {
#     'diameter': TARGET_ROTOR_DIA_M,
#     'hub_height': TARGET_HUB_HEIGHT_M,
#     'rated_power': TARGET_RATED_POWER,
# }

# df = generate_training_data(
#     wind_rose_df=wind_rose,
#     target_poly=target_poly,
#     target_centroid=target_centroid,
#     target_spec=target_spec,
#     scenarios=scenarios,
#     wd_sim=WD_SIM,
#     ws_sim=WS_SIM,
#     random_seed=RANDOM_SEED
# )

In [15]:
# Save final output
# power_kw = df["power_kw"]
# df = df.drop("power_kw", axis=1)
# df["power_kw"] = power_kw
# output_path = os.path.join(OUTPUT_DIR, 'training_data_per_turbine.csv')
# df.to_csv(output_path)
# print(df.head())

In [16]:
def parse_windfarmpoly(filename='gdf.gpkg'):
    df = gpd.read_file(DATA_DIR+'/'+filename, driver='GPKG')
    farms = {}
    for index, row in df.iterrows():
        name = row['name']
        geometry = row['geometry']
        centroid = row.geometry.centroid
        farms[name] = {'name': name,
                      'polygon': geometry,
                      'centroid_lat': centroid.y,
                      'centroid_lon': centroid.x}

    return farms
    
    

In [30]:
def find_farm(all_farms, search_name):
    search_name = search_name.lower().strip()
    if search_name in all_farms:
        return search_name, all_farms[search_name]

    for key, value in all_farms.items():
        
        if not isinstance(key, str):
            continue
        key = key.lower()
        
        if search_name == key:
            print(search_name)
            return key, value

    return None, None

In [18]:
# all_farms = parse_windfarmpoly()

# count = 0
# for farm in FARM_CONFIGS:

#     key, value = find_farm(all_farms, farm['name'])
#     if value:
#         count+=1

# print(f"{count*100/len(FARM_CONFIGS)}% of the data is fetched") 

In [19]:
test_farms = parse_windfarmpoly()
count = 0
for farm in TEST_FARM_CONFIG:
    key, value = find_farm(test_farms, farm['name'])
    if value:
        count+=1

print(f"{farm['name']} is fetched") 


horns rev i
Horns Rev I is fetched


In [20]:
def project_polygon(polygon, epsg_code):
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    gdf_projected = gdf.to_crs(epsg=epsg_code)
    return gdf_projected.geometry.iloc[0]

In [21]:
def download_era5_for_farm(farm_name, lat, lon, era5_dir, years= (2023, 2024, 2025)):
    import cdsapi
    farm_era5_dir = os.path.join(era5_dir, farm_name.lower().replace(' ','_'))
    os.makedirs(farm_era5_dir, exist_ok=True)

    files = []
    client = cdsapi.Client(quiet=True)

    for year in years:
        for month in range(1, 13):
            print(f"Downloading {year:02d} - {month:02d} data")
            file_path = os.path.join(farm_era5_dir, f'era5_{farm_name}_100m_{year}_{month}.nc')
            if os.path.exists(file_path):
                print(f"{year} - {month} already downloaded!")
            else:
                client.retrieve(
                    'reanalysis-era5-single-levels',
                    {
                        'product_type': 'reanalysis',
                        'variable': ['100m_u_component_of_wind', '100m_v_component_of_wind'],
                        'year': f'{year:02d}',
                        'month': f'{month:02d}',
                        'day': [f'{d:02d}' for d in range(1, 32)],
                        'time': [f'{h:02d}:00' for h in range(24)],
                        'area': [lat + 0.5, lon - 0.5, lat - 0.5, lon + 0.5],
                        'data_format': 'netcdf',
                    },
                    file_path
                )
        files.append(file_path)

    return files
    

In [22]:
# for farm in FARM_CONFIGS:
#     farm_name = farm['name']
#     _, farm_info = find_farm(all_farms, farm_name)
#     print(f'Downloading ERA5 for farm {farm_info['name']}')
#     download_era5_for_farm(farm_info['name'], farm_info['centroid_lat'],
#                            farm_info['centroid_lon'], ERA5_DIR)
    

In [23]:
# Test farm (Horns Rev 1 data download)
# for farm in TEST_FARM_CONFIG:
#     farm_name = farm['name']
#     _, farm_info = find_farm(test_farms, farm_name)
#     print(f'Downloading ERA5 for farm {farm_info['name']}')
#     download_era5_for_farm(farm_info['name'], farm_info['centroid_lat'],
#                            farm_info['centroid_lon'], ERA5_DIR)

In [24]:
# farms = []
# for farm in FARM_CONFIGS:
#     _, farm_info = find_farm(all_farms, farm['name'])
#     farms.append(farm_info)

In [31]:
farm = []
for i in TEST_FARM_CONFIG:
    _, farm_info = find_farm(test_farms, i['name'])
    farm.insert(0, farm_info)

print(farm)

horns rev i
[{'name': 'Horns Rev I', 'polygon': <MULTIPOLYGON (((7.796 55.503, 7.805 55.503, 7.814 55.503, 7.823 55.503, 7.8...>, 'centroid_lat': 55.48613902912691, 'centroid_lon': 7.840550917560209}]


In [32]:
# all_farm_dfs = []

# for farm_info, cfg in zip(farms, FARM_CONFIGS):
#     print(farm_info['name'])
#     farm_name = farm_info["name"]
#     farm_file_name = "_".join(farm_name.lower().split())

#     print(f"\n{'='*60}")
#     print(f"Processing: {farm_name}")
#     print(f"{'='*60}")

#     target_spec = {
#         "n_turbines": cfg["n_turbines"],
#         "diameter": cfg["rotor_dia"],
#         "hub_height": cfg["hub_height"],
#         "rated_power": cfg["rated_power"],
#         "spacing_D": cfg["spacing_D"]
#     }

#     # Load wind rose for this farm
#     wind_rose_df = load_era5_wind_rose(
#         os.path.join(ERA5_DIR, farm_file_name)
#     )

#     # FIX: Project polygon from WGS84 (degrees) to UTM (meters)
#     target_poly = project_polygon(farm_info["polygon"],  cfg["epsg"])
#     target_centroid = target_poly.centroid

#     # Generate training data
#     df_farm = generate_training_data(
#         wind_rose_df=wind_rose_df,
#         target_poly=target_poly,
#         target_centroid=target_centroid,
#         target_spec=target_spec,
#         scenarios=scenarios,
#         random_seed=RANDOM_SEED
#     )

#     df_farm["farm_name"] = farm_name

#     # Save per-farm CSV
#     output_file = os.path.join(
#         OUTPUT_DIR,
#         f"{farm_file_name}_training_data.csv"
#     )
#     df_farm.to_csv(output_file, index=False)

#     all_farm_dfs.append(df_farm)

# # Combine all farms
# if all_farm_dfs:
#     training_df = pd.concat(all_farm_dfs, ignore_index=True)
#     training_df.to_csv(
#         os.path.join(OUTPUT_DIR, "training_data_final.csv"),
#         index=False
#     )
#     print(training_df.head())

In [33]:
for farm_info, cfg in zip(farm, TEST_FARM_CONFIG):
    farm_name = farm_info["name"]
    farm_file_name = "_".join(farm_name.lower().split())

    print(f"\n{'='*60}")
    print(f"Processing: {farm_name}")
    print(f"{'='*60}")

    target_spec = {
        "n_turbines": cfg["n_turbines"],
        "diameter": cfg["rotor_dia"],
        "hub_height": cfg["hub_height"],
        "rated_power": cfg["rated_power"],
        "spacing_D": cfg["spacing_D"]
    }

    # Load wind rose for this farm
    wind_rose_df = load_era5_wind_rose(
        os.path.join(ERA5_DIR, farm_file_name)
    )

    # FIX: Project polygon from WGS84 (degrees) to UTM (meters)
    target_poly = project_polygon(farm_info["polygon"],  cfg["epsg"])
    target_centroid = target_poly.centroid

    # Generate training data
    df_farm = generate_training_data(
        wind_rose_df=wind_rose_df,
        target_poly=target_poly,
        target_centroid=target_centroid,
        target_spec=target_spec,
        scenarios=scenarios_hornsrev,
        random_seed=RANDOM_SEED
    )

    df_farm["farm_name"] = farm_name

    # Save per-farm CSV
    output_file = os.path.join(
        OUTPUT_DIR,
        f"{farm_file_name}_training_data.csv"
    )
    df_farm.to_csv(output_file, index=False)

    # test_farm_df.append(df_farm)

# Combine all farms
# if all_farm_dfs:
#     training_df = pd.concat(all_farm_dfs, ignore_index=True)
#     training_df.to_csv(
#         os.path.join(OUTPUT_DIR, "training_data_all_farms.csv"),
#         index=False
#     )
    # print(training_df.head())


Processing: Horns Rev I
Target farm: 80 turbines placed
  x range: [803172, 808308] m
  y range: [6157438, 6161542] m


In [34]:
# df = pd.read_csv(os.path.join(OUTPUT_DIR, 'training_data_final.csv'))
# power_kw = df["power_kw"]
# df = df.drop("power_kw", axis=1)
# df["power_kw"] = power_kw
# float_col = df.select_dtypes('float').columns
# df[float_col] = df[float_col].round(3)
# output_path = os.path.join(OUTPUT_DIR, 'training_data_final.csv')
# df.to_csv(output_path)
# print(df.head())

In [35]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'horns_rev_i_training_data.csv'))
power_kw = df["power_kw"]
df = df.drop("power_kw", axis=1)
df["power_kw"] = power_kw
float_col = df.select_dtypes('float').columns
df[float_col] = df[float_col].round(3)
output_path = os.path.join(OUTPUT_DIR, 'horns_rev_i_training_data.csv')
df.to_csv(output_path)
print(df.head())

   scenario_id  turbine_idx  ws_free     wd     ti  target_rotor_dia_m  \
0            0            0   14.732  240.0  0.046                  80   
1            0            1   14.732  240.0  0.046                  80   
2            0            2   14.732  240.0  0.046                  80   
3            0            3   14.732  240.0  0.046                  80   
4            0            4   14.732  240.0  0.046                  80   

   target_hub_height_m  target_rated_power_kw  nb_distance_km  \
0                   70                   2000           24.75   
1                   70                   2000           24.75   
2                   70                   2000           24.75   
3                   70                   2000           24.75   
4                   70                   2000           24.75   

   nb_direction_deg  ...  geom_lateral_dist_1st  geom_rotor_dia_1st  \
0               270  ...                 96.882                80.0   
1               270  .